<a href="https://colab.research.google.com/github/SheffieldML/GPc/blob/main/python/examples/basic_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Basic 1-D GP classification with `gpc`

CIP-0036: GPy `GPClassification` reflection on dense FTC (EP default; Laplace optional).

GPy-shaped recipe:

```python
m = gpc.GPClassification(X, Y)           # Bernoulli–probit + EP
m.optimize(iters=50)
mu, var = m.predict_noiseless(Xtest)    # latent f_*
p = m.predictive_probabilities(Xtest) # P(y=1); alias of predict_proba
```

No auto White: softness lives in EP sites / the Laplace Hessian ([CIP-0027](../../cip/cip0027.md) / [CIP-0033](../../cip/cip0033.md)). Sparse EP is out of scope; large \(N\) → native IVM.


In [ ]:
%pip install --no-cache-dir --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "gpc==0.1.0a6"


In [ ]:
%pip install matplotlib


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

print("gpc", gpc.__version__, "from", gpc.__file__)
assert gpc.__version__ >= "0.1.0a6", (
    f"need gpc>=0.1.0a6 for classification predict; got {gpc.__version__}"
)

rng = np.random.default_rng(0)
X = np.linspace(-2.0, 2.0, 40)[:, None]
Y = (np.sin(X[:, 0]) > 0.0).astype(float)[:, None]
Y = np.clip(Y + 0.0 * rng.normal(size=Y.shape), 0.0, 1.0)

m = gpc.GPClassification(X, Y, kernel="rbf", inference="ep")
m.set_by_path("rbf.lengthscale", 0.6)
m.set_by_path("rbf.variance", 1.0)
print("inference:", m.inference, "paths:", m.param_paths())
print("objective before:", m.objective())
m.optimize(iters=60)
print("objective after:", m.objective())
print(m)


In [ ]:
import matplotlib.pyplot as plt

Xtest = np.linspace(-2.5, 2.5, 200)[:, None]
mu, var = m.predict_noiseless(Xtest)
p = m.predictive_probabilities(Xtest)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X[:, 0], Y[:, 0], c="k", s=20, label="labels")
axes[0].plot(Xtest[:, 0], mu[:, 0], label="E[f_*]")
axes[0].fill_between(
    Xtest[:, 0],
    (mu - 2 * np.sqrt(var))[:, 0],
    (mu + 2 * np.sqrt(var))[:, 0],
    alpha=0.25,
)
axes[0].set_title("predict / predict_noiseless (latent f_*)")
axes[0].legend()

axes[1].scatter(X[:, 0], Y[:, 0], c="k", s=20, label="labels")
axes[1].plot(Xtest[:, 0], p[:, 0], label="P(y=1)")
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title("predictive_probabilities")
axes[1].legend()
plt.tight_layout()
plt.show()

# Optional: same data under Laplace
m_lap = gpc.GPClassification(X, Y, inference="laplace")
m_lap.set_by_path("rbf.lengthscale", 0.6)
m_lap.set_by_path("rbf.variance", 1.0)
m_lap.optimize(iters=40)
p_lap = m_lap.predict_proba(Xtest)
print("mean |p_ep - p_lap|:", float(np.mean(np.abs(p - p_lap))))
